# 10 - Federated ONE-CLASS (autoencoder) lintas-dataset (Paper 4, mitigasi)

**Hipotesis mitigasi (H-OC):** kegagalan nb03 (FedAvg SUPERVISED kolaps lintas-
dataset) berakar pada **heterogenitas PRIOR LABEL** (CIC 0,17 vs UNSW 0,55) -
dibuktikan nb09 (UNSW putus di α≈0,1). Pendekatan **one-class anomaly detection**
(ala Fed-ANIDS) menghapus ketergantungan ini: tiap klien melatih **autoencoder
HANYA pada trafik NORMAL** (y=0). Serangan dideteksi sebagai anomali
(reconstruction error > ambang). Karena 'normal' antar-dataset relatif lebih
mirip daripada batas keputusan supervised, FedAvg bobot AE SEHARUSNYA lebih stabil.

**Yang diuji:** apakah federated one-class **membalikkan** hasil nb03
(federated-SFM GLOBAL-test MCC −0,026) menjadi positif/kompetitif.

**Desain (apple-to-apple dgn nb03):**
- Ruang fitur: 9 fitur SFM (dari nb01).
- Skenario: 2 klien = {CIC, UNSW} (lintas-dataset, kasus yang GAGAL di nb03).
- Model: autoencoder ringan (numpy; ditulis mandiri di sini krn fed_node.py = MLP klasifikasi).
- Baseline: centralized-AE, local-only-AE, federated-AE (FedAvg bobot).
- Evaluasi: CIC-test, UNSW-test, GLOBAL-test (sama seperti nb03).
- Ambang τ: persentil error pada data NORMAL train (dikalibrasi, dilaporkan).

**Privasi:** tiap klien hanya kirim bobot AE; data mentah tak keluar. z-score lokal.

**Output:** `fed_out/oneclass.csv` + `oneclass_curve.csv`.

> Jalankan setelah nb01. JANGAN mengarang angka.

## 0. Setup + autoencoder numpy (one-class)

AE: 9 -> 6 -> 3 -> 6 -> 9 (ReLU di hidden, linear di output). Loss = MSE
rekonstruksi. SGD mini-batch. FedAvg bobot sama persis pola `fed_node.fedavg`.
Ditulis mandiri agar jelas untuk one-class (bukan klasifikasi).

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','pandas','scikit-learn'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, numpy as np, pandas as pd
from sklearn.metrics import (matthews_corrcoef, recall_score, precision_score,
                             f1_score, balanced_accuracy_score)
OUTDIR='fed_out'; os.makedirs(OUTDIR, exist_ok=True)
ROUNDS=100; LOCAL_EPOCHS=5; LR=1e-2; SEED=42; BATCH=256
AE_LAYERS=[9,6,3,6,9]  # autoencoder simetris

def _relu(x): return np.maximum(0,x)
def ae_init(seed=SEED):
    r=np.random.RandomState(seed); W=[]
    for a,b in zip(AE_LAYERS[:-1],AE_LAYERS[1:]):
        W.append(r.randn(a,b)*np.sqrt(2.0/a)); W.append(np.zeros(b))
    return W
def ae_forward(W, X):
    h=X; acts=[X]; pre=[]
    n=len(W)//2
    for i in range(n):
        z=h@W[2*i]+W[2*i+1]; pre.append(z)
        h=_relu(z) if i<n-1 else z   # output linear
        acts.append(h)
    return h, acts, pre
def ae_train(W, X, epochs=LOCAL_EPOCHS, lr=LR, batch=BATCH, seed=SEED):
    # SGD MSE rekonstruksi (hanya pada X normal). Backprop manual.
    r=np.random.RandomState(seed); n=len(X); W=[w.copy() for w in W]; L=len(W)//2
    for _ in range(epochs):
        idx=r.permutation(n)
        for s in range(0,n,batch):
            xb=X[idx[s:s+batch]]; m=len(xb)
            out,acts,pre=ae_forward(W,xb)
            g=(out-xb)*(2.0/m)          # dL/dout (MSE)
            grads=[None]*len(W)
            for i in reversed(range(L)):
                a_prev=acts[i]
                grads[2*i]=a_prev.T@g; grads[2*i+1]=g.sum(0)
                if i>0:
                    da=g@W[2*i].T; da=da*(pre[i-1]>0)  # relu'
                    g=da
            for i in range(len(W)): W[i]=W[i]-lr*grads[i]
    return W
def recon_err(W, X):
    out,_,_=ae_forward(W,X); return np.mean((out-X)**2, axis=1)
def fedavg(ws, sizes):
    tot=float(sum(sizes)); out=[]
    for i in range(len(ws[0])):
        out.append(sum((sizes[k]/tot)*ws[k][i] for k in range(len(ws))))
    return out
def zscore_fit(X): mu=X.mean(0); sd=X.std(0); sd[sd==0]=1.0; return mu,sd
def zscore_apply(X,mu,sd): return (X-mu)/sd
print('AE one-class siap | layers', AE_LAYERS)

## 1. Muat partisi SFM per-klien (dari nb01)

In [ ]:
def load(name):
    p=os.path.join(OUTDIR,name)
    if not os.path.exists(p): print('MISSING',p); return None,None
    d=np.load(p,allow_pickle=True); return np.asarray(d['X'],float), np.asarray(d['y'],int)
Xc_tr,yc_tr=load('cic_train.npz'); Xc_te,yc_te=load('cic_test.npz')
Xu_tr,yu_tr=load('unsw_train.npz'); Xu_te,yu_te=load('unsw_test.npz')
ok=all(v is not None for v in [Xc_tr,Xu_tr,Xc_te,Xu_te])
if ok:
    Xg_te=np.vstack([Xc_te,Xu_te]); yg_te=np.concatenate([yc_te,yu_te])
    print('CIC train',Xc_tr.shape,'normal',int((yc_tr==0).sum()))
    print('UNSW train',Xu_tr.shape,'normal',int((yu_tr==0).sum()))
    print('GLOBAL-test',Xg_te.shape,'pos',round(float(yg_te.mean()),3))
else:
    print('(partisi belum ada - jalankan nb01)')

## 2. Helper: kalibrasi ambang + evaluasi

Ambang τ dari persentil-95 error pada NORMAL train (klien sumber). Prediksi:
error > τ => serangan (1). z-score lokal klien sumber dipakai saat evaluasi
(deployment realistis: tiap node normalisasi dgn statistik lokalnya).

In [ ]:
PCTL=95  # persentil ambang pada error normal-train
def metrics(y,yp):
    return dict(MCC=round(matthews_corrcoef(y,yp),4),
                recall=round(recall_score(y,yp,zero_division=0),4),
                precision=round(precision_score(y,yp,zero_division=0),4),
                F1=round(f1_score(y,yp,zero_division=0),4),
                bal_acc=round(balanced_accuracy_score(y,yp),4))
def calib_tau(W, Xnorm_z):
    return float(np.percentile(recon_err(W, Xnorm_z), PCTL))
def predict(W, X, mu, sd, tau):
    return (recon_err(W, zscore_apply(X,mu,sd))>tau).astype(int)
print('kalibrasi + evaluasi siap | persentil ambang =', PCTL)

## 3. Baseline + federated one-class

- **centralized-AE**: latih AE pada NORMAL gabungan (CIC+UNSW) - upper bound.
- **local-only-AE**: tiap klien latih AE pada normal-nya; evaluasi silang.
- **federated-AE**: FedAvg bobot AE antar-klien (hanya normal). z-score lokal.
Ambang τ dikalibrasi dari normal-train klien sumber test (CIC-test->τ_CIC, dst);
GLOBAL pakai τ gabungan - dilaporkan apa adanya.

In [ ]:
res=[]; curve=[]
if ok:
    # data normal per klien
    Xc_n=Xc_tr[yc_tr==0]; Xu_n=Xu_tr[yu_tr==0]
    muC,sdC=zscore_fit(Xc_n); muU,sdU=zscore_fit(Xu_n)
    XcN_z=zscore_apply(Xc_n,muC,sdC); XuN_z=zscore_apply(Xu_n,muU,sdU)
    # z-score gabungan (monitoring GLOBAL)
    Xall_n=np.vstack([Xc_n,Xu_n]); muG,sdG=zscore_fit(Xall_n)
    # ---- centralized-AE ----
    Wc=ae_init(SEED); Wc=ae_train(Wc, zscore_apply(Xall_n,muG,sdG), epochs=ROUNDS*LOCAL_EPOCHS)
    tauG=calib_tau(Wc, zscore_apply(Xall_n,muG,sdG))
    for nm,(Xe,ye) in {'CIC-test':(Xc_te,yc_te),'UNSW-test':(Xu_te,yu_te),'GLOBAL-test':(Xg_te,yg_te)}.items():
        res.append({'setting':'centralized-AE','eval':nm, **metrics(ye, predict(Wc,Xe,muG,sdG,tauG))})
    # ---- local-only-AE ----
    Wlc=ae_init(SEED); Wlc=ae_train(Wlc, XcN_z, epochs=ROUNDS*LOCAL_EPOCHS); tauC=calib_tau(Wlc,XcN_z)
    Wlu=ae_init(SEED); Wlu=ae_train(Wlu, XuN_z, epochs=ROUNDS*LOCAL_EPOCHS); tauU=calib_tau(Wlu,XuN_z)
    for nm,(Xe,ye,W,mu,sd,tau) in {
        'CIC-test':(Xc_te,yc_te,Wlc,muC,sdC,tauC),'UNSW-test':(Xu_te,yu_te,Wlu,muU,sdU,tauU)}.items():
        res.append({'setting':'local-only-AE','eval':nm, **metrics(ye, predict(W,Xe,mu,sd,tau))})
    # ---- federated-AE (FedAvg bobot) ----
    Wg=ae_init(SEED); sizes=[len(Xc_n),len(Xu_n)]
    for t in range(ROUNDS):
        W1=ae_train(Wg, XcN_z, epochs=LOCAL_EPOCHS)
        W2=ae_train(Wg, XuN_z, epochs=LOCAL_EPOCHS)
        Wg=fedavg([W1,W2], sizes)
        tg=calib_tau(Wg, zscore_apply(Xall_n,muG,sdG))
        yp=predict(Wg, Xg_te, muG, sdG, tg)
        curve.append({'round':t,'MCC_global':round(matthews_corrcoef(yg_te,yp),4)})
    tauGf=calib_tau(Wg, zscore_apply(Xall_n,muG,sdG))
    tauCf=calib_tau(Wg, XcN_z); tauUf=calib_tau(Wg, XuN_z)
    for nm,(Xe,ye,mu,sd,tau) in {
        'CIC-test':(Xc_te,yc_te,muC,sdC,tauCf),'UNSW-test':(Xu_te,yu_te,muU,sdU,tauUf),
        'GLOBAL-test':(Xg_te,yg_te,muG,sdG,tauGf)}.items():
        res.append({'setting':'federated-AE','eval':nm, **metrics(ye, predict(Wg,Xe,mu,sd,tau))})
    print('one-class selesai | rounds',ROUNDS)
print('=== SEL 3 (one-class) SELESAI ===')

## 4. Simpan + bandingkan dgn nb03 (supervised)

**Kunci:** bandingkan federated-AE GLOBAL-test MCC di sini vs federated-SFM
GLOBAL-test MCC −0,026 di nb03. Bila one-class JAUH lebih baik -> mitigasi
prior-label TERBUKTI; bila tidak -> heterogenitas lebih dalam dari sekadar label.

In [ ]:
if ok and len(res):
    dfres=pd.DataFrame(res); dfres.to_csv(os.path.join(OUTDIR,'oneclass.csv'),index=False)
    pd.DataFrame(curve).to_csv(os.path.join(OUTDIR,'oneclass_curve.csv'),index=False)
    import IPython.display as ipd; ipd.display(dfres)
    g=[r for r in res if r['setting']=='federated-AE' and r['eval']=='GLOBAL-test']
    if g: print(f"\nfederated-AE GLOBAL-test MCC = {g[0]['MCC']} (bandingkan nb03 supervised: -0.026)")
else:
    print('(lewati: partisi belum ada - jalankan nb01)')
print('=== SEL 4 (hasil + banding) SELESAI ===')

## 5. Unggah ke S3 (results/oneclass)

In [ ]:
S3_BUCKET=os.environ.get('FED_S3_BUCKET','ssh-detection-features-232032302717')
REGION=os.environ.get('AWS_REGION','ap-southeast-1')
PREFIX='unsw-far/federated/results/oneclass'
try:
    import boto3
    s3=boto3.client('s3',region_name=REGION); up=0
    for f in ('oneclass.csv','oneclass_curve.csv'):
        p=os.path.join(OUTDIR,f)
        if os.path.exists(p):
            s3.upload_file(p, S3_BUCKET, f'{PREFIX}/{f}'); up+=1; print('  ok ->', f)
    print(f'\nunggah {up} berkas -> s3://{S3_BUCKET}/{PREFIX}/')
except Exception as e:
    print('(upload S3 gagal:',e,')')
print('=== SEL 5 (unggah S3) SELESAI ===')

## 6. Catatan untuk naskah

- Menguji hipotesis mitigasi: one-class (latih hanya NORMAL) menghapus
  ketergantungan prior-label yang meruntuhkan FedAvg supervised (nb03/nb09).
- Mereplikasi inti metode SOTA Fed-ANIDS (AE + reconstruction error) tetapi pada
  skenario LINTAS-DATASET (lebih sulit dari Fed-ANIDS yang homogen).
- Bila federated-AE GLOBAL-test MCC >> −0,026 (nb03): bukti bahwa paradigma
  one-class memperbaiki federasi lintas-dataset -> arah positif Paper 4.
- Ambang τ (persentil-95 normal-train) dilaporkan eksplisit; sensitivitas τ bisa
  jadi ablasi lanjutan. z-score lokal per-klien (privasi, konsisten A.1).
- JUJUR: ini AE sederhana (bukan VAE/USAD). Bila hasil menjanjikan, upgrade ke
  USAD (adversarial AE) sebagai langkah berikut.